# Batch Document Processing

Process multiple documents efficiently and collect results.

In [ ]:
import sys
sys.path.append('..')

from pathlib import Path
from PIL import Image
import pandas as pd
from tqdm import tqdm
import json

from models import ModelLoader
from utils import ImageProcessor, TextExtractor, FieldParser

## 1. Setup

In [ ]:
# Configuration
INPUT_DIR = Path('../examples')
OUTPUT_DIR = Path('./results')
OUTPUT_DIR.mkdir(exist_ok=True)

MODEL_KEY = 'got_ocr'  # or 'qwen_vl_2b'
DOCUMENT_TYPE = 'invoice'  # or 'passport', 'receipt'

In [ ]:
# Load model
model = ModelLoader.load_model(MODEL_KEY)
print(f"Model {MODEL_KEY} loaded successfully")

## 2. Collect Images

In [ ]:
# Find all image files
image_extensions = ['.jpg', '.jpeg', '.png', '.bmp']
image_files = []

for ext in image_extensions:
    image_files.extend(INPUT_DIR.rglob(f'*{ext}'))

print(f"Found {len(image_files)} images")

## 3. Process Documents

In [ ]:
results = []

for image_path in tqdm(image_files, desc="Processing documents"):
    try:
        # Load and preprocess
        image = Image.open(image_path)
        processed = ImageProcessor.preprocess(image)
        
        # Extract text
        text = model.process_image(processed)
        cleaned_text = TextExtractor.clean_text(text)
        
        # Extract fields based on document type
        if DOCUMENT_TYPE == 'invoice':
            fields = FieldParser.parse_invoice(cleaned_text)
        elif DOCUMENT_TYPE == 'passport':
            fields = FieldParser.parse_passport(cleaned_text)
        elif DOCUMENT_TYPE == 'receipt':
            fields = FieldParser.parse_receipt(cleaned_text)
        else:
            fields = {}
        
        # Calculate confidence
        confidence = TextExtractor.calculate_confidence_score(cleaned_text)
        
        # Store result
        result = {
            'filename': image_path.name,
            'text': cleaned_text,
            'confidence': confidence,
            **fields
        }
        results.append(result)
        
    except Exception as e:
        print(f"Error processing {image_path.name}: {e}")
        results.append({
            'filename': image_path.name,
            'error': str(e)
        })

print(f"\nProcessed {len(results)} documents")

## 4. Analyze Results

In [ ]:
# Create DataFrame
df = pd.DataFrame(results)

# Display summary
print("\n=== Processing Summary ===")
print(f"Total documents: {len(df)}")
print(f"Successful: {df['error'].isna().sum()}")
print(f"Failed: {df['error'].notna().sum()}")

if 'confidence' in df.columns:
    print(f"\nAverage confidence: {df['confidence'].mean():.2%}")
    print(f"Min confidence: {df['confidence'].min():.2%}")
    print(f"Max confidence: {df['confidence'].max():.2%}")

# Display first few results
print("\n=== Sample Results ===")
display(df.head())

## 5. Export Results

In [ ]:
# Export to CSV
csv_path = OUTPUT_DIR / f'results_{DOCUMENT_TYPE}_{MODEL_KEY}.csv'
df.to_csv(csv_path, index=False)
print(f"Results saved to {csv_path}")

# Export to JSON
json_path = OUTPUT_DIR / f'results_{DOCUMENT_TYPE}_{MODEL_KEY}.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f"Results saved to {json_path}")

## 6. Visualize Statistics

In [ ]:
import matplotlib.pyplot as plt

if 'confidence' in df.columns:
    # Confidence distribution
    plt.figure(figsize=(10, 6))
    plt.hist(df['confidence'].dropna(), bins=20, edgecolor='black')
    plt.xlabel('Confidence Score')
    plt.ylabel('Frequency')
    plt.title('OCR Confidence Distribution')
    plt.grid(alpha=0.3)
    plt.savefig(OUTPUT_DIR / 'confidence_distribution.png')
    plt.show()

## 7. Cleanup

In [ ]:
# Unload model
ModelLoader.unload_model(MODEL_KEY)
print("Model unloaded")